In [1]:
!pip install transformers accelerate

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [17]:
test_df = pd.read_csv("/kaggle/input/datasets/palakjaiswal24/finaldataset/test_final.csv")
train_df   = pd.read_csv("/kaggle/input/datasets/palakjaiswal24/finaldataset/train_final.csv")
val_df  = pd.read_csv("/kaggle/input/datasets/palakjaiswal24/finaldataset/val_final.csv")

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 81248
Val: 10156
Test: 10157


In [20]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

MAX_LENGTH = 256
STRIDE = 128
MAX_CHUNKS = 10


In [22]:
class HierarchicalDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=MAX_LENGTH,
            stride=STRIDE,
            truncation=True,
            padding="max_length",
            return_overflowing_tokens=True,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"]
        attention_mask = encoding["attention_mask"]

        if input_ids.size(0) > MAX_CHUNKS:
            input_ids = input_ids[:MAX_CHUNKS]
            attention_mask = attention_mask[:MAX_CHUNKS]

        pad_chunks = MAX_CHUNKS - input_ids.size(0)

        if pad_chunks > 0:
            pad_input = torch.zeros((pad_chunks, MAX_LENGTH), dtype=torch.long)
            pad_mask = torch.zeros((pad_chunks, MAX_LENGTH), dtype=torch.long)

            input_ids = torch.cat([input_ids, pad_input], dim=0)
            attention_mask = torch.cat([attention_mask, pad_mask], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": torch.tensor(label, dtype=torch.long)
        }

In [25]:
test_dataset = HierarchicalDataset(test_df, tokenizer)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [27]:
class HierarchicalXLMRBase(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained("xlm-roberta-base")
        hidden_size = 768

        self.doc_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_size,
                nhead=8,
                dim_feedforward=2048,
                dropout=0.1,
                batch_first=True
            ),
            num_layers=2
        )

        self.attention = nn.Linear(hidden_size, 1)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, input_ids, attention_mask):
        batch_size, num_chunks, seq_len = input_ids.size()

        input_ids = input_ids.view(-1, seq_len)
        attention_mask = attention_mask.view(-1, seq_len)

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        cls_embeddings = cls_embeddings.view(batch_size, num_chunks, -1)

        doc_outputs = self.doc_transformer(cls_embeddings)

        attn_weights = torch.softmax(self.attention(doc_outputs), dim=1)
        doc_rep = torch.sum(attn_weights * doc_outputs, dim=1)

        logits = self.classifier(doc_rep)

        return logits

In [28]:
model = HierarchicalXLMRBase()
model.load_state_dict(torch.load("/kaggle/input/models/palakjaiswal24/multilingual-news-detector/pytorch/default/1/best_model.pt"))
model.cuda()
model.eval()

print("Model loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.


In [29]:
def eval_epoch(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].cuda(non_blocking=True)
            attention_mask = batch["attention_mask"].cuda(non_blocking=True)
            labels = batch["label"].cuda(non_blocking=True)

            outputs = model(input_ids, attention_mask)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [30]:
test_acc = eval_epoch(model, test_loader)
print("Final Test Accuracy:", test_acc)

100%|██████████| 5079/5079 [21:22<00:00,  3.96it/s]


Final Test Accuracy: 0.9056808112631682


In [31]:
from sklearn.metrics import classification_report
import numpy as np

def detailed_eval(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].cuda()
            attention_mask = batch["attention_mask"].cuda()
            labels = batch["label"].cuda()

            outputs = model(input_ids, attention_mask)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(classification_report(all_labels, all_preds, digits=4))

In [32]:
detailed_eval(model, test_loader)

              precision    recall  f1-score   support

           0     0.8671    0.9515    0.9073      4929
           1     0.9497    0.8625    0.9040      5228

    accuracy                         0.9057     10157
   macro avg     0.9084    0.9070    0.9057     10157
weighted avg     0.9096    0.9057    0.9056     10157



In [34]:
torch.save(model.state_dict(), "final_multilingual_fake_news_model.pt")

In [35]:
model.eval()
model.cuda()

HierarchicalXLMRBase(
  (encoder): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
   

In [36]:
def predict_article(text, model, tokenizer):
    model.eval()

    # Tokenize with chunking
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        truncation=True,
        padding="max_length",
        return_overflowing_tokens=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]

    # Limit chunks
    if input_ids.size(0) > MAX_CHUNKS:
        input_ids = input_ids[:MAX_CHUNKS]
        attention_mask = attention_mask[:MAX_CHUNKS]

    # Pad chunks if fewer than MAX_CHUNKS
    pad_chunks = MAX_CHUNKS - input_ids.size(0)

    if pad_chunks > 0:
        pad_input = torch.zeros((pad_chunks, MAX_LENGTH), dtype=torch.long)
        pad_mask = torch.zeros((pad_chunks, MAX_LENGTH), dtype=torch.long)

        input_ids = torch.cat([input_ids, pad_input], dim=0)
        attention_mask = torch.cat([attention_mask, pad_mask], dim=0)

    # Add batch dimension
    input_ids = input_ids.unsqueeze(0).cuda()
    attention_mask = attention_mask.unsqueeze(0).cuda()

    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        probs = torch.softmax(outputs, dim=1)
        prediction = torch.argmax(probs, dim=1).item()

    confidence = probs[0][prediction].item()

    label_map = {0: "Real", 1: "Fake"}

    return {
        "prediction": label_map[prediction],
        "confidence": round(confidence, 4),
        "prob_real": round(probs[0][0].item(), 4),
        "prob_fake": round(probs[0][1].item(), 4)
    }

In [37]:
sample_text = """
The government announced a new economic reform policy today aimed at boosting rural employment and stabilizing inflation.
Officials stated the policy will be implemented gradually over the next quarter.
"""

result = predict_article(sample_text, model, tokenizer)
print(result)

{'prediction': 'Fake', 'confidence': 0.9928, 'prob_real': 0.0072, 'prob_fake': 0.9928}


In [38]:
fake_test = """
BREAKING: Secret government documents reveal aliens are controlling world leaders and economic policies worldwide.
"""

print(predict_article(fake_test, model, tokenizer))

{'prediction': 'Fake', 'confidence': 0.9348, 'prob_real': 0.0652, 'prob_fake': 0.9348}


In [48]:
real_test = """
चौंकाने वाला खुलासा: वैज्ञानिकों ने दावा किया है कि सरकार गुप्त रूप से मौसम को नियंत्रित कर रही है। 
सूत्रों के अनुसार, एक विशेष तकनीक के माध्यम से बारिश और सूखा कृत्रिम रूप से पैदा किया जा रहा है ताकि कृषि बाजार को प्रभावित किया जा सके। 
हालांकि सरकार ने इन आरोपों को पूरी तरह से निराधार बताया है, लेकिन सोशल मीडिया पर यह खबर तेजी से वायरल हो रही है।
"""

print(predict_article(real_test, model, tokenizer))

{'prediction': 'Fake', 'confidence': 0.8849, 'prob_real': 0.1151, 'prob_fake': 0.8849}


In [49]:
long_real_article = """
देशभर में हड़कंप: एक वायरल संदेश में दावा किया गया है कि अगले सप्ताह से सभी बैंकों में जमा धनराशि पर विशेष टैक्स लगाया जाएगा। 
कहा जा रहा है कि यह फैसला गुप्त बैठक में लिया गया है और इसकी घोषणा अचानक की जाएगी। 
सरकारी अधिकारियों ने इस खबर को भ्रामक बताया है।
"""
print(predict_article(long_real_article, model, tokenizer))

{'prediction': 'Fake', 'confidence': 0.9277, 'prob_real': 0.0723, 'prob_fake': 0.9277}
